In [5]:
# ==============================================================================
# CAPSTONE MASTER ORCHESTRATOR: BASELINE 0.2 (UNCONSTRAINED PORTFOLIO)
# ==============================================================================
import pandas as pd
import numpy as np
import itertools
from pathlib import Path
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. PARAMETRIC PORTFOLIO CONFIGURATION
# ==============================================================================
# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"

TOKENS_TO_EVALUATE = ['BTC', 'AVAX', 'BERA', 'KAITO']

EMP_START = pd.Timestamp('2025-03-01 00:00:00') 
EMP_END = pd.Timestamp('2025-12-31 23:59:00')
ALL_CALENDAR_DAYS = pd.date_range(start=EMP_START.date(), end=EMP_END.date(), freq='D').date
DAYS_TOTAL = (EMP_END - EMP_START).days

ROLLING_WINDOW_MINS = 1440  
ENTRY_THRESHOLD = 4.0      
TRADE_TIMEOUT_MINS = 300

FEES = {
    'binf': 1.70, 'okex': 1.75, 'byde': 3.20, 
    'kusw': 2.50, 'gate': 2.00, 'hype': 1.44
}

NAME_MAP = {
    'binance-futures': 'binf', 'bybit': 'byde', 'okex-swap': 'okex', 
    'gate-io-futures': 'gate', 'hyperliquid': 'hype', 'kucoin-futures': 'kusw'
}

# ==============================================================================
# 2. DETAILED STRATEGY SET-UP (DYNAMIC CONSOLE OUTPUT)
# ==============================================================================
def print_detailed_strategy_setup():
    print("======================================================================================================================")
    print(f" 📘 CAPSTONE BASELINE 0.2: MULTI-TOKEN UNCONSTRAINED EXECUTION (HYBRID FUNDING GUARDED)")
    print("======================================================================================================================")
    print(" [1] STRICTLY ROLLING EX-ANTE S-SCORE (SIGNAL ENGINE)")
    print(f"     • Computed dynamically on a {ROLLING_WINDOW_MINS}-minute rolling OLS window using shift(1) barriers.")
    print("\n [2] THE EXECUTABLE ALPHA (H*) EV GATE (ENTRY LOGIC)")
    print(f"     • Trigger: Absolute S-Score magnitude strictly >= {ENTRY_THRESHOLD}σ.")
    print("     • Net EV = Gross EV - Round_Trip_Fees - Expected Exit Slippage.")
    print("\n [3] HYBRID FUNDING EPOCH BLACKOUTS")
    print("     • Standard Venues: Epochs are inferred dynamically from 'epoch_id' shift(-1).")
    print("     • Hyperliquid (HYPE): Override applied. Mathematically enforced at XX:59 to bypass missing CSV rows.")
    print("\n [4] EXIT ROUTING MATRIX (CLOSURE MECHANICS)")
    print("     • Priority 1: Funding Evacuation (Avoids holding penalties).")
    print(f"     • Priority 2: {TRADE_TIMEOUT_MINS}m Timeout (Force close at {TRADE_TIMEOUT_MINS} minutes to recycle margin).")
    print("     • Priority 3: MA Convergence (Closes instantly when raw physical mid crosses target MA).")
    print("======================================================================================================================\n")

# ==============================================================================
# 3. NATIVE DATA UTILITIES 
# ==============================================================================
def robust_to_datetime(series):
    s_num = pd.to_numeric(series, errors='coerce')
    med_val = s_num.median()
    if pd.isna(med_val): return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_localize(None)
    unit = 'ns' if med_val > 1e16 else ('us' if med_val > 1e13 else ('ms' if med_val > 1e10 else 's'))
    return pd.to_datetime(s_num, unit=unit, errors='coerce', utc=True).dt.floor('1min').dt.tz_localize(None)

def load_aligned_bbo(df_master, exch_A, exch_B, bbo_paths):
    df_pair = pd.DataFrame(index=df_master.index)
    for exch in [exch_A, exch_B]:
        bbo = pd.read_csv(bbo_paths[exch], usecols=['minute', 'bid', 'ask'])
        bbo['timestamp'] = robust_to_datetime(bbo['minute'])
        bbo.set_index('timestamp', inplace=True)
        bbo = bbo[['bid', 'ask']].copy()
        
        bbo.loc[bbo['bid'] >= bbo['ask'], ['bid', 'ask']] = np.nan
        bbo = bbo.reindex(df_master.index).ffill(limit=3)
        df_pair[f'bid_{exch}'] = bbo['bid']
        df_pair[f'ask_{exch}'] = bbo['ask']
        df_pair[f'spread_{exch}'] = (np.log(bbo['ask']) - np.log(bbo['bid'])) * 10000

    df_pair['combined_spread'] = df_pair[f'spread_{exch_A}'] + df_pair[f'spread_{exch_B}']
    df_pair['liquidity_toll'] = df_pair['combined_spread'].rolling(1440).median().bfill()
    return df_pair

def calc_stochastic_ex_ante(df_master, adj_col, raw_col, w):
    df = pd.DataFrame(index=df_master.index)
    df[adj_col] = df_master[adj_col]
    df['raw_ma'] = df_master[raw_col].shift(1).rolling(w).mean()
    
    y_prev, x_prev = df[adj_col].shift(1), df[adj_col].shift(2)
    roll_cov = y_prev.rolling(w).cov(x_prev)
    roll_var_x = x_prev.rolling(w).var()
    roll_var_y = y_prev.rolling(w).var()
    mean_x, mean_y = x_prev.rolling(w).mean(), y_prev.rolling(w).mean()
    
    phi = (roll_cov / roll_var_x).clip(upper=0.999, lower=-0.999) 
    a = mean_y - phi * mean_x
    m = a / (1 - phi)
    var_e = (roll_var_y - (phi**2) * roll_var_x).clip(lower=1e-8)
    sigma_eq = np.sqrt(var_e / (1 - phi**2))
    
    df['s_score'] = (df[adj_col] - m) / sigma_eq
    df['kappa'] = np.where(phi > 0, -np.log(phi.clip(lower=1e-5)), 0)
    return df['s_score'].values, df['raw_ma'].values, df['kappa'].values

# ==============================================================================
# 4. CHRONOLOGICAL ENGINE LAYER
# ==============================================================================
def process_token(token):
    print(f"\n" + "="*150)
    print(f" ⚙️ INITIALIZING ENGINE FOR ASSET: {token}")
    print("="*150)
    
    master_parquet = DATA_DIR / f"{token}_TAGGED_MASTER.parquet"
    if not master_parquet.exists():
        print(f"[!] Warning: Master Parquet for {token} not found. Skipping.")
        return
        
    df_master = pd.read_parquet(master_parquet).loc[EMP_START:EMP_END]
    
    bbo_paths = {
        'binf': DATA_DIR / rf"binf_{token}_bbo.csv",
        'byde': DATA_DIR / rf"byde_{token}_bbo.csv",
        'okex': DATA_DIR / rf"okex_{token}_bbo.csv",
        'gate': DATA_DIR / rf"gate_{token}_bbo.csv",
        'hype': DATA_DIR / rf"hype_{token}_bbo.csv",
        'kusw': DATA_DIR / rf"kusw_{token}_bbo.csv"
    }
    
    exchanges = list(bbo_paths.keys())
    pairs = list(itertools.combinations(exchanges, 2))
    
    signals, bbo_storage, last_entry_t = {}, {}, {}
    n_steps = len(df_master)
    timestamps = df_master.index
    
    def get_data_driven_epoch(exch_name, full_name):
        epoch_col = f'epoch_id_{full_name}'
        if epoch_col not in df_master.columns:
            epoch_col = f'epoch_id_{exch_name}'
        if epoch_col in df_master.columns:
            shift_mask = df_master[epoch_col] != df_master[epoch_col].shift(-1)
            shift_mask.iloc[-1] = False
            return shift_mask.values, shift_mask.sum()
        return np.zeros(n_steps, dtype=bool), 0

    for exch_a, exch_b in tqdm(pairs, desc=f"Pre-computing {token} Footprints", leave=False):
        pair_label = f"{exch_a.upper()}/{exch_b.upper()}"
        full_name_A = [k for k, v in NAME_MAP.items() if v == exch_a][0]
        full_name_B = [k for k, v in NAME_MAP.items() if v == exch_b][0]
        
        adj_col = f'spread_padj_idx_mid_avg_{full_name_A}_vs_{full_name_B}'
        raw_col = f'spread_raw_mid_{full_name_A}_vs_{full_name_B}'
        
        if adj_col not in df_master.columns or raw_col not in df_master.columns: continue
            
        bbo_df = load_aligned_bbo(df_master, exch_a, exch_b, bbo_paths)
        s_score, raw_ma, kappa = calc_stochastic_ex_ante(df_master, adj_col, raw_col, ROLLING_WINDOW_MINS)
        z_exec_raw = df_master[raw_col].values
        
        mask_a = (timestamps.minute == 59) if exch_a == 'hype' else get_data_driven_epoch(exch_a, full_name_A)[0]
        mask_b = (timestamps.minute == 59) if exch_b == 'hype' else get_data_driven_epoch(exch_b, full_name_B)[0]
        is_funding_epoch = mask_a | mask_b

        signals[pair_label] = {
            's_score': s_score, 'raw_ma': raw_ma, 'raw_mid': z_exec_raw,
            'is_funding': is_funding_epoch,
            'liquidity_toll': bbo_df['liquidity_toll'].values,
            'round_trip_fee_bps': (FEES[exch_a] + FEES[exch_b]) * 2
        }
        bbo_storage[pair_label] = bbo_df
        last_entry_t[pair_label] = -60

    active_positions, trade_log = {}, []

    for t in tqdm(range(ROLLING_WINDOW_MINS, n_steps), desc=f"Processing {token} Execution"):
        current_time = timestamps[t]
        
        # ==========================================
        # 1. EVALUATE POSITION EXITS
        # ==========================================
        keys_to_remove = []
        for pair, pos in active_positions.items():
            sig_data = signals[pair]
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            hold_time = t - pos['entry_idx']
            timeout_hit = hold_time >= TRADE_TIMEOUT_MINS
            funding_hit = sig_data['is_funding'][t] 
            ma_reverted = (sig_data['raw_mid'][t] <= pos['target_ma']) if pos['direction'] == -1 else (sig_data['raw_mid'][t] >= pos['target_ma'])
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            is_bbo_invalid = np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val)
            
            must_close = False
            exit_reason = None
            
            if funding_hit: must_close, exit_reason = True, "Funding_Epoch"
            elif timeout_hit: must_close, exit_reason = True, "Timeout"
            elif ma_reverted: must_close, exit_reason = True, "MA_Hit"
            elif pos.get('force_close_pending', False): must_close, exit_reason = True, pos.get('pending_reason')
            
            if must_close:
                if is_bbo_invalid:
                    # 🟢 THE TRICK: Impute for BOTH Maximum Timeouts and Funding Evacuations
                    if hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                        valid_bbo = bbo_df.iloc[:t].dropna(subset=[
                            f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', 
                            f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'
                        ])
                        if not valid_bbo.empty:
                            last_valid = valid_bbo.iloc[-1]
                            bid_a_val, ask_a_val = last_valid[f'bid_{exch_a.lower()}'], last_valid[f'ask_{exch_a.lower()}']
                            bid_b_val, ask_b_val = last_valid[f'bid_{exch_b.lower()}'], last_valid[f'ask_{exch_b.lower()}']
                            
                            # Tag appropriately for the print matrix
                            if exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                                pos['pending_reason'] = "Funding_Imputed"
                            else:
                                pos['pending_reason'] = "Timeout_Imputed"
                        else:
                            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
                    else:
                        # Retain delaying logic ONLY for MA hits (since waiting for liquidity is fine here)
                        pos['force_close_pending'] = True
                        if 'pending_reason' not in pos:
                            pos['pending_reason'] = "MA_Hit_Delayed"
                        continue 
                
                # Assign final exit reason labels
                if pos.get('force_close_pending', False) or (is_bbo_invalid and (hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"])):
                    exit_reason = pos.get('pending_reason', "Timeout_Imputed")
                elif funding_hit: exit_reason = "Funding_Epoch"
                elif timeout_hit: exit_reason = "Timeout"
                else: exit_reason = "MA_Hit"
                
                if pos['direction'] == -1:  
                    current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                    gross_pnl = pos['z_entry_exec'] - current_z_exec
                else:                      
                    current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                    gross_pnl = current_z_exec - pos['z_entry_exec']
                
                net_pnl = gross_pnl - pos['round_trip_fee_bps']
                
                trade_log.append({
                    'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': current_time, 
                    'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': exit_reason
                })
                keys_to_remove.append(pair)
                
        for k in keys_to_remove: del active_positions[k]

        # ==========================================
        # 2. EVALUATE POSITION ENTRIES
        # ==========================================
        for pair, sig_data in signals.items():
            if pair in active_positions or t - last_entry_t[pair] < 60 or sig_data['is_funding'][t]: continue  
                
            obs_s = sig_data['s_score'][t]
            if np.isnan(obs_s) or np.abs(obs_s) < ENTRY_THRESHOLD: continue
                
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            if np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val): continue
                
            direction = -1 if obs_s > 0 else 1
            target_ma = sig_data['raw_ma'][t]
            
            if direction == -1:
                entry_phys = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                gross_ev = entry_phys - target_ma
            else:
                entry_phys = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                gross_ev = target_ma - entry_phys
                
            toll = sig_data['liquidity_toll'][t]
            net_ev = gross_ev - sig_data['round_trip_fee_bps'] - toll
            
            if net_ev <= 0: continue  
                
            active_positions[pair] = {
                'entry_idx': t, 'z_entry_exec': entry_phys, 
                'target_ma': target_ma, 'direction': direction, 
                'round_trip_fee_bps': sig_data['round_trip_fee_bps']
            }
            last_entry_t[pair] = t  

    # ==============================================================================
    # 🟢 TERMINAL LIQUIDATION BLOCK (SOLVES 912m NaN ZOMBIE HOLD BUG)
    # ==============================================================================
    terminal_idx = n_steps - 1
    terminal_time = timestamps[terminal_idx]
    
    for pair, pos in list(active_positions.items()):
        bbo_df = bbo_storage[pair]
        exch_a, exch_b = pair.split('/')
        
        # Clamp hold time strictly to avoid statistical distortion
        hold_time = min(terminal_idx - pos['entry_idx'], TRADE_TIMEOUT_MINS)
        
        valid_bbo = bbo_df.dropna(subset=[f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'])
        if not valid_bbo.empty:
            last_valid_row = valid_bbo.iloc[-1]
            bid_a_val, ask_a_val = last_valid_row[f'bid_{exch_a.lower()}'], last_valid_row[f'ask_{exch_a.lower()}']
            bid_b_val, ask_b_val = last_valid_row[f'bid_{exch_b.lower()}'], last_valid_row[f'ask_{exch_b.lower()}']
        else:
            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
            
        if pos['direction'] == -1:
            current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
            gross_pnl = pos['z_entry_exec'] - current_z_exec
        else:
            current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
            gross_pnl = current_z_exec - pos['z_entry_exec']
            
        net_pnl = gross_pnl - pos['round_trip_fee_bps']
        
        trade_log.append({
            'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': terminal_time,
            'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': "Terminal_Force_Clean"
        })

    # ==============================================================================
    # 5. FINANCIAL REPORT MATRIX WITH ADVANCED METRICS
    # ==============================================================================
    tdf = pd.DataFrame(trade_log)
    if tdf.empty: 
        print(f"\n[!] Portfolio evaluation complete for {token}: 0 trades executed.")
        return
        
    tdf['close_date'] = pd.to_datetime(tdf['Exit_Time']).dt.date

    # Aggregations
    active_daily_bps = tdf.groupby('close_date')['Net_PnL'].sum()
    calendar_daily_bps = active_daily_bps.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
    
    global_cum_bps = calendar_daily_bps.sum()
    global_peaks = calendar_daily_bps.cumsum().cummax()
    global_mdd_bps = (global_peaks - calendar_daily_bps.cumsum()).max()
    
    # Global Metrics
    cal_sharpe = (calendar_daily_bps.mean() / max(calendar_daily_bps.std(), 1e-6)) * np.sqrt(365)
    act_sharpe = (active_daily_bps.mean() / max(active_daily_bps.std(), 1e-6)) * np.sqrt(365) if len(active_daily_bps) > 1 else 0.0
    trd_sharpe = tdf['Net_PnL'].mean() / max(tdf['Net_PnL'].std(), 1e-6)
    
    gross_gains = tdf.loc[tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
    gross_losses = abs(tdf.loc[tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
    g2p_ratio = (gross_gains / gross_losses) if gross_losses > 0 else 999.99

    print("\n" + "="*150)
    print(f" 📊 {token} BASELINE 0.2: UNCONSTRAINED TEAR SHEET")
    print("="*150)
    print(" [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]")
    print(f" Total Executed Trades:     {len(tdf):<8}     | Overall Strategy Win Rate: {(tdf['Net_PnL'] > 0).mean() * 100:.2f}%")
    print(f" Cumulative PnL (BPS):      {global_cum_bps:<8.2f}     | Overall MA Reversion Rate: {tdf['Reason'].str.contains('MA').mean() * 100:.2f}%")
    print(f" Calendar Sharpe (Ann.):    {cal_sharpe:<8.2f}     | Trade-Level Sharpe (Trd):  {trd_sharpe:.2f}")
    print(f" Active Sharpe (Ann.):      {act_sharpe:<8.2f}     | Gain-to-Pain Ratio (G2P):  {g2p_ratio:.2f}")
    print(f" Max Single Trade Loss:     {tdf['Net_PnL'].min():<8.2f} bps | Max System Drawdown (BPS): {global_mdd_bps:.2f}")
    print(f" Avg. System Hold Time:     {tdf['Hold_Time'].mean():<8.1f} min   | ")
    
    print("\n [ INDIVIDUAL PAIR FINANCIAL AUDIT ]")
    print(f"{'VENUE PAIR':<12} | {'TRDS':<4} | {'WIN %':<6} | {'MA HIT%':<7} | {'AVG HLD':<7} | {'CUM RET%':<8} | {'MDD %':<7} | {'CAL SR':<6} | {'ACT SR':<6} | {'TRD SR':<6} | {'G2P':<5} | {'MAX LOSS':<9} | {'AVG NET'}")
    print("-" * 150)
    
    pair_metrics = []
    for pair in tdf['Pair'].unique():
        p_tdf = tdf[tdf['Pair'] == pair]
        trades = len(p_tdf)
        win_pct = (p_tdf['Net_PnL'] > 0).mean() * 100
        ma_hit_pct = p_tdf['Reason'].str.contains('MA').mean() * 100
        avg_hold = p_tdf['Hold_Time'].mean()
        avg_net = p_tdf['Net_PnL'].mean()
        max_loss = p_tdf['Net_PnL'].min()
        
        p_act_daily = p_tdf.groupby('close_date')['Net_PnL'].sum() / 10000
        p_cal_daily = p_act_daily.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
        cum_ret = p_cal_daily.sum() * 100
        ann_ret = (cum_ret / DAYS_TOTAL) * 365.0
        
        c_sr = (p_cal_daily.mean() / max(p_cal_daily.std(), 1e-6)) * np.sqrt(365)
        a_sr = (p_act_daily.mean() / max(p_act_daily.std(), 1e-6)) * np.sqrt(365) if len(p_act_daily) > 1 else 0.0
        t_sr = p_tdf['Net_PnL'].mean() / max(p_tdf['Net_PnL'].std(), 1e-6)
        
        g_wins = p_tdf.loc[p_tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
        g_loss = abs(p_tdf.loc[p_tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
        pair_g2p = (g_wins / g_loss) if g_loss > 0 else 999.99
        
        p_cum_sum = p_cal_daily.cumsum() * 100
        mdd = (p_cum_sum.cummax() - p_cum_sum).max()
        
        pair_metrics.append({
            'Pair': pair, 'Trades': trades, 'Win%': win_pct, 'MA%': ma_hit_pct, 
            'Hold': avg_hold, 'CumRet': cum_ret, 'AnnRet': ann_ret, 'MDD': mdd, 
            'CalSR': c_sr, 'ActSR': a_sr, 'TrdSR': t_sr, 'G2P': pair_g2p, 
            'MaxLoss': max_loss, 'AvgNet': avg_net
        })
        
    pair_metrics = sorted(pair_metrics, key=lambda x: x['CumRet'], reverse=True)
    for pm in pair_metrics:
        print(f"{pm['Pair']:<12} | {pm['Trades']:<4.0f} | {pm['Win%']:<5.1f}% | {pm['MA%']:<6.1f}% | {pm['Hold']:<7.1f} | {pm['CumRet']:<7.2f}% | {pm['MDD']:<6.2f}% | {pm['CalSR']:<6.2f} | {pm['ActSR']:<6.2f} | {pm['TrdSR']:<6.2f} | {pm['G2P']:<5.2f} | {pm['MaxLoss']:<9.2f} | {pm['AvgNet']:.2f}")

    print("\n [ TERMINATION ROUTING COUNT ]")
    print(f" Normal MA Hit: {len(tdf[tdf['Reason']=='MA_Hit']):<5} | Delayed MA Hit: {len(tdf[tdf['Reason']=='MA_Hit_Delayed']):<5}")
    print(f" Normal Fundng: {len(tdf[tdf['Reason']=='Funding_Epoch']):<5} | Imputed Fundng: {len(tdf[tdf['Reason']=='Funding_Imputed']):<5}")
    print(f" Normal Timout: {len(tdf[tdf['Reason']=='Timeout']):<5} | Imputed Timout: {len(tdf[tdf['Reason']=='Timeout_Imputed']):<5}")
    term_cleans = len(tdf[tdf['Reason']=='Terminal_Force_Clean'])
    if term_cleans > 0: print(f" Terminal Clean: {term_cleans:<5} | (End of Year NaN Dump)")

    print("\n--- HOLD TIME DIAGNOSTIC BY EXIT REASON ---")
    hold_breakdown = tdf.groupby('Reason')['Hold_Time'].agg(['mean', 'count'])
    print(hold_breakdown)
    print("="*150 + "\n")

if __name__ == "__main__":
    print_detailed_strategy_setup()
    for asset in TOKENS_TO_EVALUATE:
        process_token(asset)

 📘 CAPSTONE BASELINE 0.2: MULTI-TOKEN UNCONSTRAINED EXECUTION (HYBRID FUNDING GUARDED)
 [1] STRICTLY ROLLING EX-ANTE S-SCORE (SIGNAL ENGINE)
     • Computed dynamically on a 1440-minute rolling OLS window using shift(1) barriers.

 [2] THE EXECUTABLE ALPHA (H*) EV GATE (ENTRY LOGIC)
     • Trigger: Absolute S-Score magnitude strictly >= 4.0σ.
     • Net EV = Gross EV - Round_Trip_Fees - Expected Exit Slippage.

 [3] HYBRID FUNDING EPOCH BLACKOUTS
     • Standard Venues: Epochs are inferred dynamically from 'epoch_id' shift(-1).
     • Hyperliquid (HYPE): Override applied. Mathematically enforced at XX:59 to bypass missing CSV rows.

 [4] EXIT ROUTING MATRIX (CLOSURE MECHANICS)
     • Priority 1: Funding Evacuation (Avoids holding penalties).
     • Priority 2: 300m Timeout (Force close at 300 minutes to recycle margin).
     • Priority 3: MA Convergence (Closes instantly when raw physical mid crosses target MA).


 ⚙️ INITIALIZING ENGINE FOR ASSET: BTC


Pre-computing BTC Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

Processing BTC Execution:   0%|          | 0/439200 [00:00<?, ?it/s]


 📊 BTC BASELINE 0.2: UNCONSTRAINED TEAR SHEET
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     117          | Overall Strategy Win Rate: 44.44%
 Cumulative PnL (BPS):      432.66       | Overall MA Reversion Rate: 38.46%
 Calendar Sharpe (Ann.):    2.01         | Trade-Level Sharpe (Trd):  0.18
 Active Sharpe (Ann.):      5.08         | Gain-to-Pain Ratio (G2P):  2.53
 Max Single Trade Loss:     -19.55   bps | Max System Drawdown (BPS): 57.92
 Avg. System Hold Time:     31.5     min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P   | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
BINF/HYPE    | 44   | 40.9 % | 29.5  % | 26.7    | 1.88   % | 0.34  % | 1.19   | 3.42   | 0.16   | 2.98  | -6.89     | 4.27
BINF/GATE    | 14   | 64.3 

Pre-computing AVAX Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

...

Processing BERA Execution:   0%|          | 0/439200 [00:00<?, ?it/s]


 📊 BERA BASELINE 0.2: UNCONSTRAINED TEAR SHEET
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     127          | Overall Strategy Win Rate: 59.06%
 Cumulative PnL (BPS):      1498.57      | Overall MA Reversion Rate: 52.76%
 Calendar Sharpe (Ann.):    1.76         | Trade-Level Sharpe (Trd):  0.19
 Active Sharpe (Ann.):      4.73         | Gain-to-Pain Ratio (G2P):  2.20
 Max Single Trade Loss:     -86.31   bps | Max System Drawdown (BPS): 473.77
 Avg. System Hold Time:     36.2     min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P   | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
GATE/HYPE    | 14   | 85.7 % | 78.6  % | 16.9    | 8.68   % | 0.19  % | 1.72   | 8.53   | 0.45   | 24.52 | -18.57    | 62.01
BINF/HYPE    | 31   | 61

Pre-computing KAITO Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

Processing KAITO Execution:   0%|          | 0/439200 [00:00<?, ?it/s]


 📊 KAITO BASELINE 0.2: UNCONSTRAINED TEAR SHEET
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     307          | Overall Strategy Win Rate: 53.09%
 Cumulative PnL (BPS):      -1896.78     | Overall MA Reversion Rate: 43.00%
 Calendar Sharpe (Ann.):    -1.57        | Trade-Level Sharpe (Trd):  -0.12
 Active Sharpe (Ann.):      -3.32        | Gain-to-Pain Ratio (G2P):  0.64
 Max Single Trade Loss:     -444.61  bps | Max System Drawdown (BPS): 1896.78
 Avg. System Hold Time:     53.8     min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P   | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
BINF/HYPE    | 47   | 63.8 % | 63.8  % | 22.7    | 2.50   % | 1.24  % | 2.21   | 6.60   | 0.30   | 2.22  | -44.06    | 5.33
BYDE/HYPE    | 15   | 

In [8]:
# ==============================================================================
# BASELINE 0.2: MULTI-TOKEN EXECUTION (ILLIQUIDITY ENTRY GUARDED)
# WITH DEFERRED ROUTING / LOOKBACK IMPUTATION FOR EXIT GAPS & TERMINAL LIQUIDATION
# ==============================================================================
import pandas as pd
import numpy as np
import itertools
from pathlib import Path
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. PARAMETRIC PORTFOLIO CONFIGURATION
# ==============================================================================
# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"
TOKENS_TO_EVALUATE = ['BTC', 'AVAX', 'BERA', 'KAITO']

EMP_START = pd.Timestamp('2025-03-01 00:00:00') 
EMP_END = pd.Timestamp('2025-12-31 23:59:00')
ALL_CALENDAR_DAYS = pd.date_range(start=EMP_START.date(), end=EMP_END.date(), freq='D').date
DAYS_TOTAL = (EMP_END - EMP_START).days

# --- MICROSTRUCTURE RISK GUARD ---
MAX_ENTRY_SPREAD_BPS = 10.0    # Prevent entry if combined order books are too wide

ROLLING_WINDOW_MINS = 1440  
ENTRY_THRESHOLD = 4.0      
TRADE_TIMEOUT_MINS = 300

FEES = {
    'binf': 1.70, 'okex': 1.75, 'byde': 3.20, 
    'kusw': 2.50, 'gate': 2.00, 'hype': 1.44
}

NAME_MAP = {
    'binance-futures': 'binf', 'bybit': 'byde', 'okex-swap': 'okex', 
    'gate-io-futures': 'gate', 'hyperliquid': 'hype', 'kucoin-futures': 'kusw'
}

# ==============================================================================
# 2. DETAILED STRATEGY SET-UP (DYNAMIC CONSOLE OUTPUT)
# ==============================================================================
def print_detailed_strategy_setup():
    print("======================================================================================================================")
    print(f" 📘 CAPSTONE BASELINE 0.2: MULTI-TOKEN INDEPENDENT PAIR EXECUTION (ILLIQUIDITY GUARDED)")
    print("======================================================================================================================")
    print(" [1] STRICTLY ROLLING EX-ANTE S-SCORE (SIGNAL ENGINE)")
    print(f"     • Computed dynamically on a {ROLLING_WINDOW_MINS}-minute rolling OLS window using shift(1) barriers.")
    print("\n [2] THE EXECUTABLE ALPHA (H*) EV GATE (ENTRY LOGIC)")
    print(f"     • Trigger: Absolute S-Score magnitude strictly >= {ENTRY_THRESHOLD}σ.")
    print("     • Net EV = Gross EV - Round_Trip_Fees - Expected Exit Slippage.")
    print("\n [3] MICROSTRUCTURE RISK MANAGEMENT (ENTRY PROTECTION)")
    print(f"     • Illiquidity Entry Guard: Enforced Ceiling at {MAX_ENTRY_SPREAD_BPS} bps.")
    print("     • Dynamically calculates the sum of the physical bid-ask spreads for both execution legs.")
    print("     • Rejects trade entry if structural widening points to order book flatlining or data anomalies.")
    print("     • Note: Intraday stop-losses have been discarded to prevent malicious phantom spread stop-outs.")
    print("\n [4] HYBRID FUNDING EPOCH BLACKOUTS")
    print("     • Standard Venues: Epochs are inferred dynamically from 'epoch_id' shift(-1).")
    print("     • Hyperliquid (HYPE): Override applied. Mathematically enforced at XX:59 to bypass missing CSV rows.")
    print("\n [5] EXIT ROUTING MATRIX (CLOSURE MECHANICS)")
    print("     • Priority 1: Funding Evacuation (Avoids holding penalties).")
    print(f"     • Priority 2: {TRADE_TIMEOUT_MINS}m Timeout (Force close at {TRADE_TIMEOUT_MINS} minutes to recycle margin).")
    print("     • Priority 3: MA Convergence (Closes instantly when raw physical mid crosses target MA).")
    print("======================================================================================================================\n")

# ==============================================================================
# 3. NATIVE DATA UTILITIES 
# ==============================================================================
def robust_to_datetime(series):
    s_num = pd.to_numeric(series, errors='coerce')
    med_val = s_num.median()
    if pd.isna(med_val): return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_localize(None)
    unit = 'ns' if med_val > 1e16 else ('us' if med_val > 1e13 else ('ms' if med_val > 1e10 else 's'))
    return pd.to_datetime(s_num, unit=unit, errors='coerce', utc=True).dt.floor('1min').dt.tz_localize(None)

def load_aligned_bbo(df_master, exch_A, exch_B, bbo_paths):
    df_pair = pd.DataFrame(index=df_master.index)
    for exch in [exch_A, exch_B]:
        bbo = pd.read_csv(bbo_paths[exch], usecols=['minute', 'bid', 'ask'])
        bbo['timestamp'] = robust_to_datetime(bbo['minute'])
        bbo.set_index('timestamp', inplace=True)
        bbo = bbo[['bid', 'ask']].copy()
        
        bbo.loc[bbo['bid'] >= bbo['ask'], ['bid', 'ask']] = np.nan
        bbo = bbo.reindex(df_master.index).ffill(limit=3)
        df_pair[f'bid_{exch}'] = bbo['bid']
        df_pair[f'ask_{exch}'] = bbo['ask']
        df_pair[f'spread_{exch}'] = (np.log(bbo['ask']) - np.log(bbo['bid'])) * 10000

    df_pair['combined_spread'] = df_pair[f'spread_{exch_A}'] + df_pair[f'spread_{exch_B}']
    df_pair['liquidity_toll'] = df_pair['combined_spread'].rolling(1440).median().bfill()
    return df_pair

def calc_stochastic_ex_ante(df_master, adj_col, raw_col, w):
    df = pd.DataFrame(index=df_master.index)
    df[adj_col] = df_master[adj_col]
    df['raw_ma'] = df_master[raw_col].shift(1).rolling(w).mean()
    
    y_prev, x_prev = df[adj_col].shift(1), df[adj_col].shift(2)
    roll_cov = y_prev.rolling(w).cov(x_prev)
    roll_var_x = x_prev.rolling(w).var()
    roll_var_y = y_prev.rolling(w).var()
    mean_x, mean_y = x_prev.rolling(w).mean(), y_prev.rolling(w).mean()
    
    phi = (roll_cov / roll_var_x).clip(upper=0.999, lower=-0.999) 
    a = mean_y - phi * mean_x
    m = a / (1 - phi)
    var_e = (roll_var_y - (phi**2) * roll_var_x).clip(lower=1e-8)
    sigma_eq = np.sqrt(var_e / (1 - phi**2))
    
    df['s_score'] = (df[adj_col] - m) / sigma_eq
    df['kappa'] = np.where(phi > 0, -np.log(phi.clip(lower=1e-5)), 0)
    return df['s_score'].values, df['raw_ma'].values, df['kappa'].values

# ==============================================================================
# 4. CHRONOLOGICAL ENGINE LAYER
# ==============================================================================
def process_token(token):
    print(f"\n" + "="*150)
    print(f" ⚙️ INITIALIZING ENGINE FOR ASSET: {token}")
    print("="*150)
    
    master_parquet = DATA_DIR / f"{token}_TAGGED_MASTER.parquet"
    if not master_parquet.exists():
        print(f"[!] Warning: Master Parquet for {token} not found. Skipping.")
        return
        
    df_master = pd.read_parquet(master_parquet).loc[EMP_START:EMP_END]
    
    bbo_paths = {
        'binf': DATA_DIR / rf"binf_{token}_bbo.csv",
        'byde': DATA_DIR / rf"byde_{token}_bbo.csv",
        'okex': DATA_DIR / rf"okex_{token}_bbo.csv",
        'gate': DATA_DIR / rf"gate_{token}_bbo.csv",
        'hype': DATA_DIR / rf"hype_{token}_bbo.csv",
        'kusw': DATA_DIR / rf"kusw_{token}_bbo.csv"
    }
    
    exchanges = list(bbo_paths.keys())
    pairs = list(itertools.combinations(exchanges, 2))
    
    signals, bbo_storage, last_entry_t = {}, {}, {}
    n_steps = len(df_master)
    timestamps = df_master.index
    
    def get_data_driven_epoch(exch_name, full_name):
        epoch_col = f'epoch_id_{full_name}'
        if epoch_col not in df_master.columns:
            epoch_col = f'epoch_id_{exch_name}'
        if epoch_col in df_master.columns:
            shift_mask = df_master[epoch_col] != df_master[epoch_col].shift(-1)
            shift_mask.iloc[-1] = False
            return shift_mask.values, shift_mask.sum()
        return np.zeros(n_steps, dtype=bool), 0

    for exch_a, exch_b in tqdm(pairs, desc=f"Pre-computing {token} Footprints", leave=False):
        pair_label = f"{exch_a.upper()}/{exch_b.upper()}"
        full_name_A = [k for k, v in NAME_MAP.items() if v == exch_a][0]
        full_name_B = [k for k, v in NAME_MAP.items() if v == exch_b][0]
        
        adj_col = f'spread_padj_idx_mid_avg_{full_name_A}_vs_{full_name_B}'
        raw_col = f'spread_raw_mid_{full_name_A}_vs_{full_name_B}'
        
        if adj_col not in df_master.columns or raw_col not in df_master.columns: continue
            
        bbo_df = load_aligned_bbo(df_master, exch_a, exch_b, bbo_paths)
        s_score, raw_ma, kappa = calc_stochastic_ex_ante(df_master, adj_col, raw_col, ROLLING_WINDOW_MINS)
        z_exec_raw = df_master[raw_col].values
        
        mask_a = (timestamps.minute == 59) if exch_a == 'hype' else get_data_driven_epoch(exch_a, full_name_A)[0]
        mask_b = (timestamps.minute == 59) if exch_b == 'hype' else get_data_driven_epoch(exch_b, full_name_B)[0]
        is_funding_epoch = mask_a | mask_b

        signals[pair_label] = {
            's_score': s_score, 'raw_ma': raw_ma, 'raw_mid': z_exec_raw,
            'is_funding': is_funding_epoch,
            'liquidity_toll': bbo_df['liquidity_toll'].values,
            'round_trip_fee_bps': (FEES[exch_a] + FEES[exch_b]) * 2
        }
        bbo_storage[pair_label] = bbo_df
        last_entry_t[pair_label] = -60

    active_positions, trade_log = {}, []

    for t in tqdm(range(ROLLING_WINDOW_MINS, n_steps), desc=f"Processing {token} Execution"):
        current_time = timestamps[t]
        
        # ==========================================
        # 1. EVALUATE POSITION EXITS (DEFERRED ROUTING + LOOKBACK IMPUTATION)
        # ==========================================
        keys_to_remove = []
        for pair, pos in active_positions.items():
            sig_data = signals[pair]
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            hold_time = t - pos['entry_idx']
            timeout_hit = hold_time >= TRADE_TIMEOUT_MINS
            funding_hit = sig_data['is_funding'][t] 
            ma_reverted = (sig_data['raw_mid'][t] <= pos['target_ma']) if pos['direction'] == -1 else (sig_data['raw_mid'][t] >= pos['target_ma'])
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            is_bbo_invalid = np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val)
            
            must_close = False
            exit_reason = None
            
            if funding_hit: must_close, exit_reason = True, "Funding_Epoch"
            elif timeout_hit: must_close, exit_reason = True, "Timeout"
            elif ma_reverted: must_close, exit_reason = True, "MA_Hit"
            elif pos.get('force_close_pending', False): must_close, exit_reason = True, pos.get('pending_reason')
            
            if must_close:
                if is_bbo_invalid:
                    # Impute for Maximum Timeouts and Funding Evacuations
                    if hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                        valid_bbo = bbo_df.iloc[:t].dropna(subset=[
                            f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', 
                            f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'
                        ])
                        if not valid_bbo.empty:
                            last_valid = valid_bbo.iloc[-1]
                            bid_a_val, ask_a_val = last_valid[f'bid_{exch_a.lower()}'], last_valid[f'ask_{exch_a.lower()}']
                            bid_b_val, ask_b_val = last_valid[f'bid_{exch_b.lower()}'], last_valid[f'ask_{exch_b.lower()}']
                            
                            if exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                                pos['pending_reason'] = "Funding_Imputed"
                            else:
                                pos['pending_reason'] = "Timeout_Imputed"
                        else:
                            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
                    else:
                        # Defer and wait for liquidity on organic MA cross reversions
                        pos['force_close_pending'] = True
                        if 'pending_reason' not in pos:
                            pos['pending_reason'] = "MA_Hit_Delayed"
                        continue 
                
                # Lock terminal categorization tracking string
                if pos.get('force_close_pending', False) or (is_bbo_invalid and (hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"])):
                    exit_reason = pos.get('pending_reason', "Timeout_Imputed")
                elif funding_hit: exit_reason = "Funding_Epoch"
                elif timeout_hit: exit_reason = "Timeout"
                else: exit_reason = "MA_Hit"
                
                if pos['direction'] == -1:  
                    current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                    gross_pnl = pos['z_entry_exec'] - current_z_exec
                else:                      
                    current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                    gross_pnl = current_z_exec - pos['z_entry_exec']
                
                net_pnl = gross_pnl - pos['round_trip_fee_bps']
                
                trade_log.append({
                    'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': current_time, 
                    'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': exit_reason
                })
                keys_to_remove.append(pair)
                
        for k in keys_to_remove: del active_positions[k]

        # ==========================================
        # 2. EVALUATE POSITION ENTRIES (ILLIQUIDITY GUARDED)
        # ==========================================
        for pair, sig_data in signals.items():
            if pair in active_positions or t - last_entry_t[pair] < 60 or sig_data['is_funding'][t]: continue  
                
            obs_s = sig_data['s_score'][t]
            if np.isnan(obs_s) or np.abs(obs_s) < ENTRY_THRESHOLD: continue
                
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            if np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val): continue
                
            # 🛡️ THE ILLIQUIDITY ENTRY GUARD 
            curr_spread_a = (np.log(ask_a_val) - np.log(bid_a_val)) * 10000
            curr_spread_b = (np.log(ask_b_val) - np.log(bid_b_val)) * 10000
            if (curr_spread_a + curr_spread_b) > MAX_ENTRY_SPREAD_BPS:
                continue

            direction = -1 if obs_s > 0 else 1
            target_ma = sig_data['raw_ma'][t]
            
            if direction == -1:
                entry_phys = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                gross_ev = entry_phys - target_ma
            else:
                entry_phys = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                gross_ev = target_ma - entry_phys
                
            toll = sig_data['liquidity_toll'][t]
            net_ev = gross_ev - sig_data['round_trip_fee_bps'] - toll
            
            if net_ev <= 0: continue  
                
            active_positions[pair] = {
                'entry_idx': t, 'z_entry_exec': entry_phys, 
                'target_ma': target_ma, 'direction': direction, 
                'round_trip_fee_bps': sig_data['round_trip_fee_bps']
            }
            last_entry_t[pair] = t  

    # ==============================================================================
    # 🟢 TERMINAL LIQUIDATION BLOCK (End of Year Cutoff)
    # ==============================================================================
    terminal_idx = n_steps - 1
    terminal_time = timestamps[terminal_idx]
    
    for pair, pos in list(active_positions.items()):
        bbo_df = bbo_storage[pair]
        exch_a, exch_b = pair.split('/')
        
        hold_time = min(terminal_idx - pos['entry_idx'], TRADE_TIMEOUT_MINS)
        
        valid_bbo = bbo_df.dropna(subset=[f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'])
        if not valid_bbo.empty:
            last_valid_row = valid_bbo.iloc[-1]
            bid_a_val, ask_a_val = last_valid_row[f'bid_{exch_a.lower()}'], last_valid_row[f'ask_{exch_a.lower()}']
            bid_b_val, ask_b_val = last_valid_row[f'bid_{exch_b.lower()}'], last_valid_row[f'ask_{exch_b.lower()}']
        else:
            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
            
        if pos['direction'] == -1:
            current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
            gross_pnl = pos['z_entry_exec'] - current_z_exec
        else:
            current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
            gross_pnl = current_z_exec - pos['z_entry_exec']
            
        net_pnl = gross_pnl - pos['round_trip_fee_bps']
        
        trade_log.append({
            'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': terminal_time,
            'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': "Terminal_Force_Clean"
        })

    # ==============================================================================
    # 5. FINANCIAL REPORT MATRIX WITH ADVANCED METRICS
    # ==============================================================================
    tdf = pd.DataFrame(trade_log)
    if tdf.empty: 
        print(f"\n[!] Portfolio evaluation complete for {token}: 0 trades executed.")
        return
        
    tdf['close_date'] = pd.to_datetime(tdf['Exit_Time']).dt.date

    # Aggregations
    active_daily_bps = tdf.groupby('close_date')['Net_PnL'].sum()
    calendar_daily_bps = active_daily_bps.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
    
    global_cum_bps = calendar_daily_bps.sum()
    global_peaks = calendar_daily_bps.cumsum().cummax()
    global_mdd_bps = (global_peaks - calendar_daily_bps.cumsum()).max()
    
    # Global Metrics
    cal_sharpe = (calendar_daily_bps.mean() / max(calendar_daily_bps.std(), 1e-6)) * np.sqrt(365)
    act_sharpe = (active_daily_bps.mean() / max(active_daily_bps.std(), 1e-6)) * np.sqrt(365) if len(active_daily_bps) > 1 else 0.0
    
    g_trd_std = tdf['Net_PnL'].std()
    trd_sharpe = (tdf['Net_PnL'].mean() / g_trd_std) if pd.notna(g_trd_std) and g_trd_std > 1e-6 else 0.0
    
    gross_gains = tdf.loc[tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
    gross_losses = abs(tdf.loc[tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
    g2p_ratio = (gross_gains / gross_losses) if gross_losses > 0 else 999.99

    print("\n" + "="*150)
    print(f" 📊 {token} BASELINE 0.2: ILLIQUIDITY GUARDED TEAR SHEET (Max Spread = {MAX_ENTRY_SPREAD_BPS} bps)")
    print("="*150)
    print(" [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]")
    print(f" Total Executed Trades:     {len(tdf):<8}     | Overall Strategy Win Rate: {(tdf['Net_PnL'] > 0).mean() * 100:.2f}%")
    print(f" Cumulative PnL (BPS):      {global_cum_bps:<8.2f}     | Overall MA Reversion Rate: {tdf['Reason'].str.contains('MA').mean() * 100:.2f}%")
    print(f" Calendar Sharpe (Ann.):    {cal_sharpe:<8.2f}     | Trade-Level Sharpe (Trd):  {trd_sharpe:.2f}")
    print(f" Active Sharpe (Ann.):      {act_sharpe:<8.2f}     | Gain-to-Pain Ratio (G2P):  {g2p_ratio:.2f}")
    print(f" Max Single Trade Loss:     {tdf['Net_PnL'].min():<8.2f} bps | Max System Drawdown (BPS): {global_mdd_bps:.2f}")
    print(f" Avg. System Hold Time:     {tdf['Hold_Time'].mean():<8.1f} min   | ")
    
    print("\n [ INDIVIDUAL PAIR FINANCIAL AUDIT ]")
    print(f"{'VENUE PAIR':<12} | {'TRDS':<4} | {'WIN %':<6} | {'MA HIT%':<7} | {'AVG HLD':<7} | {'CUM RET%':<8} | {'MDD %':<7} | {'CAL SR':<6} | {'ACT SR':<6} | {'TRD SR':<6} | {'G2P':<6} | {'MAX LOSS':<9} | {'AVG NET'}")
    print("-" * 150)
    
    pair_metrics = []
    for pair in tdf['Pair'].unique():
        p_tdf = tdf[tdf['Pair'] == pair]
        trades = len(p_tdf)
        win_pct = (p_tdf['Net_PnL'] > 0).mean() * 100
        ma_hit_pct = p_tdf['Reason'].str.contains('MA').mean() * 100
        avg_hold = p_tdf['Hold_Time'].mean()
        avg_net = p_tdf['Net_PnL'].mean()
        max_loss = p_tdf['Net_PnL'].min()
        
        p_act_daily = p_tdf.groupby('close_date')['Net_PnL'].sum() / 10000
        p_cal_daily = p_act_daily.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
        cum_ret = p_cal_daily.sum() * 100
        ann_ret = (cum_ret / DAYS_TOTAL) * 365.0
        
        c_sr = (p_cal_daily.mean() / max(p_cal_daily.std(), 1e-6)) * np.sqrt(365)
        a_sr = (p_act_daily.mean() / max(p_act_daily.std(), 1e-6)) * np.sqrt(365) if len(p_act_daily) > 1 else 0.0
        
        t_sr_std = p_tdf['Net_PnL'].std()
        t_sr = (p_tdf['Net_PnL'].mean() / t_sr_std) if pd.notna(t_sr_std) and t_sr_std > 1e-6 else 0.0
        
        g_wins = p_tdf.loc[p_tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
        g_loss = abs(p_tdf.loc[p_tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
        pair_g2p = (g_wins / g_loss) if g_loss > 0 else 999.99
        
        p_cum_sum = p_cal_daily.cumsum() * 100
        mdd = (p_cum_sum.cummax() - p_cum_sum).max()
        
        pair_metrics.append({
            'Pair': pair, 'Trades': trades, 'Win%': win_pct, 'MA%': ma_hit_pct, 
            'Hold': avg_hold, 'CumRet': cum_ret, 'AnnRet': ann_ret, 'MDD': mdd, 
            'CalSR': c_sr, 'ActSR': a_sr, 'TrdSR': t_sr, 'G2P': pair_g2p, 
            'MaxLoss': max_loss, 'AvgNet': avg_net
        })
        
    pair_metrics = sorted(pair_metrics, key=lambda x: x['CumRet'], reverse=True)
    for pm in pair_metrics:
        print(f"{pm['Pair']:<12} | {pm['Trades']:<4.0f} | {pm['Win%']:<5.1f}% | {pm['MA%']:<6.1f}% | {pm['Hold']:<7.1f} | {pm['CumRet']:<7.2f}% | {pm['MDD']:<6.2f}% | {pm['CalSR']:<6.2f} | {pm['ActSR']:<6.2f} | {pm['TrdSR']:<6.2f} | {pm['G2P']:<6.2f} | {pm['MaxLoss']:<9.2f} | {pm['AvgNet']:.2f}")

    print("\n [ TERMINATION ROUTING COUNT ]")
    print(f" Normal MA Hit: {len(tdf[tdf['Reason']=='MA_Hit']):<5} | Delayed MA Hit: {len(tdf[tdf['Reason']=='MA_Hit_Delayed']):<5}")
    print(f" Normal Fundng: {len(tdf[tdf['Reason']=='Funding_Epoch']):<5} | Imputed Fundng: {len(tdf[tdf['Reason']=='Funding_Imputed']):<5}")
    print(f" Normal Timout: {len(tdf[tdf['Reason']=='Timeout']):<5} | Imputed Timout: {len(tdf[tdf['Reason']=='Timeout_Imputed']):<5}")
    term_cleans = len(tdf[tdf['Reason']=='Terminal_Force_Clean'])
    if term_cleans > 0: print(f" Termnl Force C: {term_cleans:<5} | (End of Year Cutoff)")

    print("\n--- HOLD TIME DIAGNOSTIC BY EXIT REASON ---")
    hold_breakdown = tdf.groupby('Reason')['Hold_Time'].agg(['mean', 'count'])
    print(hold_breakdown)
    print("="*150 + "\n")

if __name__ == "__main__":
    print_detailed_strategy_setup()
    for asset in TOKENS_TO_EVALUATE:
        process_token(asset)

 📘 CAPSTONE BASELINE 0.2: MULTI-TOKEN INDEPENDENT PAIR EXECUTION (ILLIQUIDITY GUARDED)
 [1] STRICTLY ROLLING EX-ANTE S-SCORE (SIGNAL ENGINE)
     • Computed dynamically on a 1440-minute rolling OLS window using shift(1) barriers.

 [2] THE EXECUTABLE ALPHA (H*) EV GATE (ENTRY LOGIC)
     • Trigger: Absolute S-Score magnitude strictly >= 4.0σ.
     • Net EV = Gross EV - Round_Trip_Fees - Expected Exit Slippage.

 [3] MICROSTRUCTURE RISK MANAGEMENT (ENTRY PROTECTION)
     • Illiquidity Entry Guard: Enforced Ceiling at 10.0 bps.
     • Dynamically calculates the sum of the physical bid-ask spreads for both execution legs.
     • Rejects trade entry if structural widening points to order book flatlining or data anomalies.
     • Note: Intraday stop-losses have been discarded to prevent malicious phantom spread stop-outs.

 [4] HYBRID FUNDING EPOCH BLACKOUTS
     • Standard Venues: Epochs are inferred dynamically from 'epoch_id' shift(-1).
     • Hyperliquid (HYPE): Override applied. Mathem

Pre-computing BTC Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

Processing BTC Execution:   0%|          | 0/439200 [00:00<?, ?it/s]


 📊 BTC BASELINE 0.2: ILLIQUIDITY GUARDED TEAR SHEET (Max Spread = 10.0 bps)
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     117          | Overall Strategy Win Rate: 44.44%
 Cumulative PnL (BPS):      415.58       | Overall MA Reversion Rate: 38.46%
 Calendar Sharpe (Ann.):    2.00         | Trade-Level Sharpe (Trd):  0.18
 Active Sharpe (Ann.):      5.06         | Gain-to-Pain Ratio (G2P):  2.47
 Max Single Trade Loss:     -19.55   bps | Max System Drawdown (BPS): 57.92
 Avg. System Hold Time:     31.6     min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P    | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
BINF/HYPE    | 44   | 40.9 % | 29.5  % | 27.0    | 1.71   % | 0.34  % | 1.19   | 3.44   | 0.16   | 2.80   | -6.89     | 

Pre-computing AVAX Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

...

Processing BERA Execution:   0%|          | 0/439200 [00:00<?, ?it/s]


 📊 BERA BASELINE 0.2: ILLIQUIDITY GUARDED TEAR SHEET (Max Spread = 10.0 bps)
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     71           | Overall Strategy Win Rate: 57.75%
 Cumulative PnL (BPS):      690.56       | Overall MA Reversion Rate: 45.07%
 Calendar Sharpe (Ann.):    1.32         | Trade-Level Sharpe (Trd):  0.25
 Active Sharpe (Ann.):      3.88         | Gain-to-Pain Ratio (G2P):  2.32
 Max Single Trade Loss:     -51.89   bps | Max System Drawdown (BPS): 169.84
 Avg. System Hold Time:     39.9     min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P    | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
BINF/HYPE    | 24   | 58.3 % | 50.0  % | 22.3    | 3.11   % | 0.47  % | 1.89   | 7.21   | 0.37   | 4.16   | -15.82    

Pre-computing KAITO Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

Processing KAITO Execution:   0%|          | 0/439200 [00:00<?, ?it/s]


 📊 KAITO BASELINE 0.2: ILLIQUIDITY GUARDED TEAR SHEET (Max Spread = 10.0 bps)
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     201          | Overall Strategy Win Rate: 58.71%
 Cumulative PnL (BPS):      -120.48      | Overall MA Reversion Rate: 45.77%
 Calendar Sharpe (Ann.):    -0.14        | Trade-Level Sharpe (Trd):  -0.02
 Active Sharpe (Ann.):      -0.31        | Gain-to-Pain Ratio (G2P):  0.94
 Max Single Trade Loss:     -377.85  bps | Max System Drawdown (BPS): 951.71
 Avg. System Hold Time:     48.8     min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P    | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
GATE/HYPE    | 16   | 81.2 % | 62.5  % | 21.8    | 2.88   % | 0.21  % | 2.68   | 14.94  | 0.75   | 11.59  | -17.31  

In [10]:
# ==============================================================================
# PER-TOKEN CONFIGURABLE PARAMETRIC EXECUTION FOR OUT-OF-SAMPLE TESTS
# ==============================================================================
import pandas as pd
import numpy as np
import itertools
from pathlib import Path
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. PARAMETRIC PORTFOLIO CONFIGURATION
# ==============================================================================
# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"

EMP_START = pd.Timestamp('2026-01-01 00:00:00') 
EMP_END = pd.Timestamp('2026-02-28 23:59:00')
ALL_CALENDAR_DAYS = pd.date_range(start=EMP_START.date(), end=EMP_END.date(), freq='D').date
DAYS_TOTAL = (EMP_END - EMP_START).days

# --- ASSET-SPECIFIC HYPERPARAMETERS (SIGMA & MICROSTRUCTURE GUARDS) ---
TOKEN_CONFIGS = {
    'BTC': {
        'entry_threshold': 4.0,          # Sigma threshold
        'max_entry_spread_bps': 10.0      # Illiquidity guard ceiling
    },
    'AVAX': {
        'entry_threshold': 4.0,
        'max_entry_spread_bps': 20.0
    },
    'BERA': {
        'entry_threshold': 4.0,
        'max_entry_spread_bps': 10.0
    },
    'KAITO': {
        'entry_threshold': 4.0,
        'max_entry_spread_bps': 5.0
    }
}

ROLLING_WINDOW_MINS = 1440  
TRADE_TIMEOUT_MINS = 300

FEES = {
    'binf': 1.70, 'okex': 1.75, 'byde': 3.20, 
    'kusw': 2.50, 'gate': 2.00, 'hype': 1.44
}

NAME_MAP = {
    'binance-futures': 'binf', 'bybit': 'byde', 'okex-swap': 'okex', 
    'gate-io-futures': 'gate', 'hyperliquid': 'hype', 'kucoin-futures': 'kusw'
}

# ==============================================================================
# 2. DETAILED STRATEGY SET-UP (DYNAMIC CONSOLE OUTPUT)
# ==============================================================================
def print_detailed_strategy_setup():
    print("======================================================================================================================")
    print(f" 📘 CAPSTONE BASELINE 0.2.1: ASSET-CUSTOMIZABLE PAIR EXECUTION (PER-TOKEN ROUTING)")
    print("======================================================================================================================")
    print(" [1] STRICTLY ROLLING EX-ANTE S-SCORE (SIGNAL ENGINE)")
    print(f"     • Computed dynamically on a {ROLLING_WINDOW_MINS}-minute rolling OLS window using shift(1) barriers.")
    print("\n [2] THE EXECUTABLE ALPHA (H*) EV GATE (ENTRY LOGIC)")
    print("     • Trigger: Absolute S-Score magnitude strictly >= Asset-Specific Sigma Threshold (σ).")
    print("     • Net EV = Gross EV - Round_Trip_Fees - Expected Exit Slippage.")
    print("\n [3] MICROSTRUCTURE RISK MANAGEMENT (ENTRY PROTECTION)")
    print("     • Illiquidity Entry Guard: Enforced dynamically using per-token spread bounds.")
    print("     • Dynamically calculates the sum of the physical bid-ask spreads for both execution legs.")
    print("     • Rejects trade entry if structural widening points to order book flatlining or data anomalies.")
    print("     • Note: Intraday stop-losses have been discarded to prevent malicious phantom spread stop-outs.")
    print("\n [4] HYBRID FUNDING EPOCH BLACKOUTS")
    print("     • Standard Venues: Epochs are inferred dynamically from 'epoch_id' shift(-1).")
    print("     • Hyperliquid (HYPE): Override applied. Mathematically enforced at XX:59 to bypass missing CSV rows.")
    print("\n [5] EXIT ROUTING MATRIX (CLOSURE MECHANICS)")
    print("     • Priority 1: Funding Evacuation (Avoids holding penalties).")
    print(f"     • Priority 2: {TRADE_TIMEOUT_MINS}m Timeout (Force close at {TRADE_TIMEOUT_MINS} minutes to recycle margin).")
    print("     • Priority 3: MA Convergence (Closes instantly when raw physical mid crosses target MA).")
    print("======================================================================================================================\n")

# ==============================================================================
# 3. NATIVE DATA UTILITIES 
# ==============================================================================
def robust_to_datetime(series):
    s_num = pd.to_numeric(series, errors='coerce')
    med_val = s_num.median()
    if pd.isna(med_val): return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_localize(None)
    unit = 'ns' if med_val > 1e16 else ('us' if med_val > 1e13 else ('ms' if med_val > 1e10 else 's'))
    return pd.to_datetime(s_num, unit=unit, errors='coerce', utc=True).dt.floor('1min').dt.tz_localize(None)

def load_aligned_bbo(df_master, exch_A, exch_B, bbo_paths):
    df_pair = pd.DataFrame(index=df_master.index)
    for exch in [exch_A, exch_B]:
        bbo = pd.read_csv(bbo_paths[exch], usecols=['minute', 'bid', 'ask'])
        bbo['timestamp'] = robust_to_datetime(bbo['minute'])
        bbo.set_index('timestamp', inplace=True)
        bbo = bbo[['bid', 'ask']].copy()
        
        bbo.loc[bbo['bid'] >= bbo['ask'], ['bid', 'ask']] = np.nan
        bbo = bbo.reindex(df_master.index).ffill(limit=3)
        df_pair[f'bid_{exch}'] = bbo['bid']
        df_pair[f'ask_{exch}'] = bbo['ask']
        df_pair[f'spread_{exch}'] = (np.log(bbo['ask']) - np.log(bbo['bid'])) * 10000

    df_pair['combined_spread'] = df_pair[f'spread_{exch_A}'] + df_pair[f'spread_{exch_B}']
    df_pair['liquidity_toll'] = df_pair['combined_spread'].rolling(1440).median().bfill()
    return df_pair

def calc_stochastic_ex_ante(df_master, adj_col, raw_col, w):
    df = pd.DataFrame(index=df_master.index)
    df[adj_col] = df_master[adj_col]
    df['raw_ma'] = df_master[raw_col].shift(1).rolling(w).mean()
    
    y_prev, x_prev = df[adj_col].shift(1), df[adj_col].shift(2)
    roll_cov = y_prev.rolling(w).cov(x_prev)
    roll_var_x = x_prev.rolling(w).var()
    roll_var_y = y_prev.rolling(w).var()
    mean_x, mean_y = x_prev.rolling(w).mean(), y_prev.rolling(w).mean()
    
    phi = (roll_cov / roll_var_x).clip(upper=0.999, lower=-0.999) 
    a = mean_y - phi * mean_x
    m = a / (1 - phi)
    var_e = (roll_var_y - (phi**2) * roll_var_x).clip(lower=1e-8)
    sigma_eq = np.sqrt(var_e / (1 - phi**2))
    
    df['s_score'] = (df[adj_col] - m) / sigma_eq
    df['kappa'] = np.where(phi > 0, -np.log(phi.clip(lower=1e-5)), 0)
    return df['s_score'].values, df['raw_ma'].values, df['kappa'].values

# ==============================================================================
# 4. CHRONOLOGICAL ENGINE LAYER
# ==============================================================================
def process_token(token, config):
    # Unpack asset-specific variables
    entry_threshold = config['entry_threshold']
    max_entry_spread_bps = config['max_entry_spread_bps']

    print(f"\n" + "="*150)
    print(f" ⚙️ INITIALIZING ENGINE FOR ASSET: {token} | PARAMETERS: Target Sigma = {entry_threshold}σ, Max Spread Guard = {max_entry_spread_bps} bps")
    print("="*150)
    
    master_parquet = DATA_DIR / f"{token}_TAGGED_MASTER.parquet"
    if not master_parquet.exists():
        print(f"[!] Warning: Master Parquet for {token} not found. Skipping.")
        return
        
    df_master = pd.read_parquet(master_parquet).loc[EMP_START:EMP_END]
    
    bbo_paths = {
        'binf': DATA_DIR / rf"binf_{token}_bbo.csv",
        'byde': DATA_DIR / rf"byde_{token}_bbo.csv",
        'okex': DATA_DIR / rf"okex_{token}_bbo.csv",
        'gate': DATA_DIR / rf"gate_{token}_bbo.csv",
        'hype': DATA_DIR / rf"hype_{token}_bbo.csv",
        'kusw': DATA_DIR / rf"kusw_{token}_bbo.csv"
    }
    
    exchanges = list(bbo_paths.keys())
    pairs = list(itertools.combinations(exchanges, 2))
    
    signals, bbo_storage, last_entry_t = {}, {}, {}
    n_steps = len(df_master)
    timestamps = df_master.index
    
    def get_data_driven_epoch(exch_name, full_name):
        epoch_col = f'epoch_id_{full_name}'
        if epoch_col not in df_master.columns:
            epoch_col = f'epoch_id_{exch_name}'
        if epoch_col in df_master.columns:
            shift_mask = df_master[epoch_col] != df_master[epoch_col].shift(-1)
            shift_mask.iloc[-1] = False
            return shift_mask.values, shift_mask.sum()
        return np.zeros(n_steps, dtype=bool), 0

    for exch_a, exch_b in tqdm(pairs, desc=f"Pre-computing {token} Footprints", leave=False):
        pair_label = f"{exch_a.upper()}/{exch_b.upper()}"
        full_name_A = [k for k, v in NAME_MAP.items() if v == exch_a][0]
        full_name_B = [k for k, v in NAME_MAP.items() if v == exch_b][0]
        
        adj_col = f'spread_padj_idx_mid_avg_{full_name_A}_vs_{full_name_B}'
        raw_col = f'spread_raw_mid_{full_name_A}_vs_{full_name_B}'
        
        if adj_col not in df_master.columns or raw_col not in df_master.columns: continue
            
        bbo_df = load_aligned_bbo(df_master, exch_a, exch_b, bbo_paths)
        s_score, raw_ma, kappa = calc_stochastic_ex_ante(df_master, adj_col, raw_col, ROLLING_WINDOW_MINS)
        z_exec_raw = df_master[raw_col].values
        
        mask_a = (timestamps.minute == 59) if exch_a == 'hype' else get_data_driven_epoch(exch_a, full_name_A)[0]
        mask_b = (timestamps.minute == 59) if exch_b == 'hype' else get_data_driven_epoch(exch_b, full_name_B)[0]
        is_funding_epoch = mask_a | mask_b

        signals[pair_label] = {
            's_score': s_score, 'raw_ma': raw_ma, 'raw_mid': z_exec_raw,
            'is_funding': is_funding_epoch,
            'liquidity_toll': bbo_df['liquidity_toll'].values,
            'round_trip_fee_bps': (FEES[exch_a] + FEES[exch_b]) * 2
        }
        bbo_storage[pair_label] = bbo_df
        last_entry_t[pair_label] = -60

    active_positions, trade_log = {}, []

    for t in tqdm(range(ROLLING_WINDOW_MINS, n_steps), desc=f"Processing {token} Execution"):
        current_time = timestamps[t]
        
        # ==========================================
        # 1. EVALUATE POSITION EXITS (DEFERRED ROUTING + LOOKBACK IMPUTATION)
        # ==========================================
        keys_to_remove = []
        for pair, pos in active_positions.items():
            sig_data = signals[pair]
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            hold_time = t - pos['entry_idx']
            timeout_hit = hold_time >= TRADE_TIMEOUT_MINS
            funding_hit = sig_data['is_funding'][t] 
            ma_reverted = (sig_data['raw_mid'][t] <= pos['target_ma']) if pos['direction'] == -1 else (sig_data['raw_mid'][t] >= pos['target_ma'])
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            is_bbo_invalid = np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val)
            
            must_close = False
            exit_reason = None
            
            if funding_hit: must_close, exit_reason = True, "Funding_Epoch"
            elif timeout_hit: must_close, exit_reason = True, "Timeout"
            elif ma_reverted: must_close, exit_reason = True, "MA_Hit"
            elif pos.get('force_close_pending', False): must_close, exit_reason = True, pos.get('pending_reason')
            
            if must_close:
                if is_bbo_invalid:
                    # Impute for Maximum Timeouts and Funding Evacuations
                    if hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                        valid_bbo = bbo_df.iloc[:t].dropna(subset=[
                            f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', 
                            f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'
                        ])
                        if not valid_bbo.empty:
                            last_valid = valid_bbo.iloc[-1]
                            bid_a_val, ask_a_val = last_valid[f'bid_{exch_a.lower()}'], last_valid[f'ask_{exch_a.lower()}']
                            bid_b_val, ask_b_val = last_valid[f'bid_{exch_b.lower()}'], last_valid[f'ask_{exch_b.lower()}']
                            
                            if exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                                pos['pending_reason'] = "Funding_Imputed"
                            else:
                                pos['pending_reason'] = "Timeout_Imputed"
                        else:
                            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
                    else:
                        # Defer and wait for liquidity on organic MA cross reversions
                        pos['force_close_pending'] = True
                        if 'pending_reason' not in pos:
                            pos['pending_reason'] = "MA_Hit_Delayed"
                        continue 
                
                # Lock terminal categorization tracking string
                if pos.get('force_close_pending', False) or (is_bbo_invalid and (hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"])):
                    exit_reason = pos.get('pending_reason', "Timeout_Imputed")
                elif funding_hit: exit_reason = "Funding_Epoch"
                elif timeout_hit: exit_reason = "Timeout"
                else: exit_reason = "MA_Hit"
                
                if pos['direction'] == -1:  
                    current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                    gross_pnl = pos['z_entry_exec'] - current_z_exec
                else:                      
                    current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                    gross_pnl = current_z_exec - pos['z_entry_exec']
                
                net_pnl = gross_pnl - pos['round_trip_fee_bps']
                
                trade_log.append({
                    'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': current_time, 
                    'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': exit_reason
                })
                keys_to_remove.append(pair)
                
        for k in keys_to_remove: del active_positions[k]

        # ==========================================
        # 2. EVALUATE POSITION ENTRIES (ILLIQUIDITY GUARDED)
        # ==========================================
        for pair, sig_data in signals.items():
            if pair in active_positions or t - last_entry_t[pair] < 60 or sig_data['is_funding'][t]: continue  
                
            obs_s = sig_data['s_score'][t]
            if np.isnan(obs_s) or np.abs(obs_s) < entry_threshold: continue
                
            bbo_df = bbo_storage[pair]
            exch_a, exch_b = pair.split('/')
            
            bid_a_val, ask_a_val = bbo_df[f'bid_{exch_a.lower()}'].iloc[t], bbo_df[f'ask_{exch_a.lower()}'].iloc[t]
            bid_b_val, ask_b_val = bbo_df[f'bid_{exch_b.lower()}'].iloc[t], bbo_df[f'ask_{exch_b.lower()}'].iloc[t]
            if np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val): continue
                
            # 🛡️ THE ILLIQUIDITY ENTRY GUARD (Dynamic Config Specific)
            curr_spread_a = (np.log(ask_a_val) - np.log(bid_a_val)) * 10000
            curr_spread_b = (np.log(ask_b_val) - np.log(bid_b_val)) * 10000
            if (curr_spread_a + curr_spread_b) > max_entry_spread_bps:
                continue

            direction = -1 if obs_s > 0 else 1
            target_ma = sig_data['raw_ma'][t]
            
            if direction == -1:
                entry_phys = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                gross_ev = entry_phys - target_ma
            else:
                entry_phys = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                gross_ev = target_ma - entry_phys
                
            toll = sig_data['liquidity_toll'][t]
            net_ev = gross_ev - sig_data['round_trip_fee_bps'] - toll
            
            if net_ev <= 0: continue  
                
            active_positions[pair] = {
                'entry_idx': t, 'z_entry_exec': entry_phys, 
                'target_ma': target_ma, 'direction': direction, 
                'round_trip_fee_bps': sig_data['round_trip_fee_bps']
            }
            last_entry_t[pair] = t  

    # ==============================================================================
    # 🟢 TERMINAL LIQUIDATION BLOCK (End of Year Cutoff)
    # ==============================================================================
    terminal_idx = n_steps - 1
    terminal_time = timestamps[terminal_idx]
    
    for pair, pos in list(active_positions.items()):
        bbo_df = bbo_storage[pair]
        exch_a, exch_b = pair.split('/')
        
        hold_time = min(terminal_idx - pos['entry_idx'], TRADE_TIMEOUT_MINS)
        
        valid_bbo = bbo_df.dropna(subset=[f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'])
        if not valid_bbo.empty:
            last_valid_row = valid_bbo.iloc[-1]
            bid_a_val, ask_a_val = last_valid_row[f'bid_{exch_a.lower()}'], last_valid_row[f'ask_{exch_a.lower()}']
            bid_b_val, ask_b_val = last_valid_row[f'bid_{exch_b.lower()}'], last_valid_row[f'ask_{exch_b.lower()}']
        else:
            bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
            
        if pos['direction'] == -1:
            current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
            gross_pnl = pos['z_entry_exec'] - current_z_exec
        else:
            current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
            gross_pnl = current_z_exec - pos['z_entry_exec']
            
        net_pnl = gross_pnl - pos['round_trip_fee_bps']
        
        trade_log.append({
            'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': terminal_time,
            'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': "Terminal_Force_Clean"
        })

    # ==============================================================================
    # 5. FINANCIAL REPORT MATRIX WITH ADVANCED METRICS
    # ==============================================================================
    tdf = pd.DataFrame(trade_log)
    if tdf.empty: 
        print(f"\n[!] Portfolio evaluation complete for {token}: 0 trades executed.")
        return
        
    tdf['close_date'] = pd.to_datetime(tdf['Exit_Time']).dt.date

    # Aggregations
    active_daily_bps = tdf.groupby('close_date')['Net_PnL'].sum()
    calendar_daily_bps = active_daily_bps.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
    
    global_cum_bps = calendar_daily_bps.sum()
    global_peaks = calendar_daily_bps.cumsum().cummax()
    global_mdd_bps = (global_peaks - calendar_daily_bps.cumsum()).max()
    
    # Global Metrics
    cal_sharpe = (calendar_daily_bps.mean() / max(calendar_daily_bps.std(), 1e-6)) * np.sqrt(365)
    act_sharpe = (active_daily_bps.mean() / max(active_daily_bps.std(), 1e-6)) * np.sqrt(365) if len(active_daily_bps) > 1 else 0.0
    
    g_trd_std = tdf['Net_PnL'].std()
    trd_sharpe = (tdf['Net_PnL'].mean() / g_trd_std) if pd.notna(g_trd_std) and g_trd_std > 1e-6 else 0.0
    
    gross_gains = tdf.loc[tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
    gross_losses = abs(tdf.loc[tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
    g2p_ratio = (gross_gains / gross_losses) if gross_losses > 0 else 999.99

    print("\n" + "="*150)
    print(f" 📊 {token} BASELINE 0.2.1 TEAR SHEET | CONFIG: (Max Spread Constraint = {max_entry_spread_bps} bps, Sigma = {entry_threshold}σ)")
    print("="*150)
    print(" [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]")
    print(f" Total Executed Trades:     {len(tdf):<8}     | Overall Strategy Win Rate: {(tdf['Net_PnL'] > 0).mean() * 100:.2f}%")
    print(f" Cumulative PnL (BPS):      {global_cum_bps:<8.2f}     | Overall MA Reversion Rate: {tdf['Reason'].str.contains('MA').mean() * 100:.2f}%")
    print(f" Calendar Sharpe (Ann.):    {cal_sharpe:<8.2f}     | Trade-Level Sharpe (Trd):  {trd_sharpe:.2f}")
    print(f" Active Sharpe (Ann.):      {act_sharpe:<8.2f}     | Gain-to-Pain Ratio (G2P):  {g2p_ratio:.2f}")
    print(f" Max Single Trade Loss:     {tdf['Net_PnL'].min():<8.2f} bps | Max System Drawdown (BPS): {global_mdd_bps:.2f}")
    print(f" Avg. System Hold Time:     {tdf['Hold_Time'].mean():<8.1f} min   | ")
    
    print("\n [ INDIVIDUAL PAIR FINANCIAL AUDIT ]")
    print(f"{'VENUE PAIR':<12} | {'TRDS':<4} | {'WIN %':<6} | {'MA HIT%':<7} | {'AVG HLD':<7} | {'CUM RET%':<8} | {'MDD %':<7} | {'CAL SR':<6} | {'ACT SR':<6} | {'TRD SR':<6} | {'G2P':<6} | {'MAX LOSS':<9} | {'AVG NET'}")
    print("-" * 150)
    
    pair_metrics = []
    for pair in tdf['Pair'].unique():
        p_tdf = tdf[tdf['Pair'] == pair]
        trades = len(p_tdf)
        win_pct = (p_tdf['Net_PnL'] > 0).mean() * 100
        ma_hit_pct = p_tdf['Reason'].str.contains('MA').mean() * 100
        avg_hold = p_tdf['Hold_Time'].mean()
        avg_net = p_tdf['Net_PnL'].mean()
        max_loss = p_tdf['Net_PnL'].min()
        
        p_act_daily = p_tdf.groupby('close_date')['Net_PnL'].sum() / 10000
        p_cal_daily = p_act_daily.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
        cum_ret = p_cal_daily.sum() * 100
        ann_ret = (cum_ret / DAYS_TOTAL) * 365.0
        
        c_sr = (p_cal_daily.mean() / max(p_cal_daily.std(), 1e-6)) * np.sqrt(365)
        a_sr = (p_act_daily.mean() / max(p_act_daily.std(), 1e-6)) * np.sqrt(365) if len(p_act_daily) > 1 else 0.0
        
        t_sr_std = p_tdf['Net_PnL'].std()
        t_sr = (p_tdf['Net_PnL'].mean() / t_sr_std) if pd.notna(t_sr_std) and t_sr_std > 1e-6 else 0.0
        
        g_wins = p_tdf.loc[p_tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
        g_loss = abs(p_tdf.loc[p_tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
        pair_g2p = (g_wins / g_loss) if g_loss > 0 else 999.99
        
        p_cum_sum = p_cal_daily.cumsum() * 100
        mdd = (p_cum_sum.cummax() - p_cum_sum).max()
        
        pair_metrics.append({
            'Pair': pair, 'Trades': trades, 'Win%': win_pct, 'MA%': ma_hit_pct, 
            'Hold': avg_hold, 'CumRet': cum_ret, 'AnnRet': ann_ret, 'MDD': mdd, 
            'CalSR': c_sr, 'ActSR': a_sr, 'TrdSR': t_sr, 'G2P': pair_g2p, 
            'MaxLoss': max_loss, 'AvgNet': avg_net
        })
        
    pair_metrics = sorted(pair_metrics, key=lambda x: x['CumRet'], reverse=True)
    for pm in pair_metrics:
        print(f"{pm['Pair']:<12} | {pm['Trades']:<4.0f} | {pm['Win%']:<5.1f}% | {pm['MA%']:<6.1f}% | {pm['Hold']:<7.1f} | {pm['CumRet']:<7.2f}% | {pm['MDD']:<6.2f}% | {pm['CalSR']:<6.2f} | {pm['ActSR']:<6.2f} | {pm['TrdSR']:<6.2f} | {pm['G2P']:<6.2f} | {pm['MaxLoss']:<9.2f} | {pm['AvgNet']:.2f}")

    print("\n [ TERMINATION ROUTING COUNT ]")
    print(f" Normal MA Hit: {len(tdf[tdf['Reason']=='MA_Hit']):<5} | Delayed MA Hit: {len(tdf[tdf['Reason']=='MA_Hit_Delayed']):<5}")
    print(f" Normal Fundng: {len(tdf[tdf['Reason']=='Funding_Epoch']):<5} | Imputed Fundng: {len(tdf[tdf['Reason']=='Funding_Imputed']):<5}")
    print(f" Normal Timout: {len(tdf[tdf['Reason']=='Timeout']):<5} | Imputed Timout: {len(tdf[tdf['Reason']=='Timeout_Imputed']):<5}")
    term_cleans = len(tdf[tdf['Reason']=='Terminal_Force_Clean'])
    if term_cleans > 0: print(f" Termnl Force C: {term_cleans:<5} | (End of Year Cutoff)")

    print("\n--- HOLD TIME DIAGNOSTIC BY EXIT REASON ---")
    hold_breakdown = tdf.groupby('Reason')['Hold_Time'].agg(['mean', 'count'])
    print(hold_breakdown)
    print("="*150 + "\n")

if __name__ == "__main__":
    print_detailed_strategy_setup()
    for asset, configuration in TOKEN_CONFIGS.items():
        process_token(asset, configuration)

 📘 CAPSTONE BASELINE 0.2.1: ASSET-CUSTOMIZABLE PAIR EXECUTION (PER-TOKEN ROUTING)
 [1] STRICTLY ROLLING EX-ANTE S-SCORE (SIGNAL ENGINE)
     • Computed dynamically on a 1440-minute rolling OLS window using shift(1) barriers.

 [2] THE EXECUTABLE ALPHA (H*) EV GATE (ENTRY LOGIC)
     • Trigger: Absolute S-Score magnitude strictly >= Asset-Specific Sigma Threshold (σ).
     • Net EV = Gross EV - Round_Trip_Fees - Expected Exit Slippage.

 [3] MICROSTRUCTURE RISK MANAGEMENT (ENTRY PROTECTION)
     • Illiquidity Entry Guard: Enforced dynamically using per-token spread bounds.
     • Dynamically calculates the sum of the physical bid-ask spreads for both execution legs.
     • Rejects trade entry if structural widening points to order book flatlining or data anomalies.
     • Note: Intraday stop-losses have been discarded to prevent malicious phantom spread stop-outs.

 [4] HYBRID FUNDING EPOCH BLACKOUTS
     • Standard Venues: Epochs are inferred dynamically from 'epoch_id' shift(-1).
    

Pre-computing BTC Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

Processing BTC Execution:   0%|          | 0/46080 [00:00<?, ?it/s]


 📊 BTC BASELINE 0.2.1 TEAR SHEET | CONFIG: (Max Spread Constraint = 10.0 bps, Sigma = 4.0σ)
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     12           | Overall Strategy Win Rate: 58.33%
 Cumulative PnL (BPS):      18.90        | Overall MA Reversion Rate: 50.00%
 Calendar Sharpe (Ann.):    2.19         | Trade-Level Sharpe (Trd):  0.33
 Active Sharpe (Ann.):      7.34         | Gain-to-Pain Ratio (G2P):  2.82
 Max Single Trade Loss:     -5.14    bps | Max System Drawdown (BPS): 5.65
 Avg. System Hold Time:     25.7     min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P    | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
BINF/OKEX    | 2    | 100.0% | 100.0 % | 9.0     | 0.18   % | 0.00  % | 3.10   | 24.73  | 1.29   | 999.99

Pre-computing AVAX Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

...

Processing BERA Execution:   0%|          | 0/46080 [00:00<?, ?it/s]


 📊 BERA BASELINE 0.2.1 TEAR SHEET | CONFIG: (Max Spread Constraint = 10.0 bps, Sigma = 4.0σ)
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     23           | Overall Strategy Win Rate: 78.26%
 Cumulative PnL (BPS):      336.88       | Overall MA Reversion Rate: 65.22%
 Calendar Sharpe (Ann.):    4.26         | Trade-Level Sharpe (Trd):  0.49
 Active Sharpe (Ann.):      11.43        | Gain-to-Pain Ratio (G2P):  5.55
 Max Single Trade Loss:     -20.22   bps | Max System Drawdown (BPS): 17.51
 Avg. System Hold Time:     46.0     min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P    | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
BINF/OKEX    | 6    | 83.3 % | 83.3  % | 64.7    | 1.92   % | 0.02  % | 3.91   | 16.45  | 0.76   | 112.

Pre-computing KAITO Footprints:   0%|          | 0/15 [00:00<?, ?it/s]

Processing KAITO Execution:   0%|          | 0/46080 [00:00<?, ?it/s]


 📊 KAITO BASELINE 0.2.1 TEAR SHEET | CONFIG: (Max Spread Constraint = 5.0 bps, Sigma = 4.0σ)
 [ GLOBAL AGGREGATE TRACK METRICS (Unscaled Basis Points) ]
 Total Executed Trades:     3            | Overall Strategy Win Rate: 33.33%
 Cumulative PnL (BPS):      36.03        | Overall MA Reversion Rate: 0.00%
 Calendar Sharpe (Ann.):    1.84         | Trade-Level Sharpe (Trd):  0.39
 Active Sharpe (Ann.):      7.39         | Gain-to-Pain Ratio (G2P):  4.05
 Max Single Trade Loss:     -7.06    bps | Max System Drawdown (BPS): 7.06
 Avg. System Hold Time:     208.0    min   | 

 [ INDIVIDUAL PAIR FINANCIAL AUDIT ]
VENUE PAIR   | TRDS | WIN %  | MA HIT% | AVG HLD | CUM RET% | MDD %   | CAL SR | ACT SR | TRD SR | G2P    | MAX LOSS  | AVG NET
------------------------------------------------------------------------------------------------------------------------------------------------------
BINF/OKEX    | 3    | 33.3 % | 0.0   % | 208.0   | 0.36   % | 0.07  % | 1.84   | 7.39   | 0.39   | 4.05  